# Graph + FGW per generator model, without folds

Everything is computed **separately for each LLM that generated the videos**: the examples and the evaluated videos always come from the same model.

**No cross-validation split.** A fixed split decides which videos serve as examples, and the results turned out to depend on it. Instead, for each model, label (human or reference) and number of examples $M$:

1. a **draw** takes $M$ random videos per level (Low, Medium, High) among the videos of the model, and computes the 3 FGW barycenters (as many nodes as the median of their graphs);
2. every video **not in the draw** gets the level of the nearest barycenter;
3. this is repeated `N_DRAWS` times.

For a given video, the examples are thus a uniform random set of $M$ videos per level among the *other* videos of its model, which is the same as drawing $M$ examples for each sentence, but each draw's barycenters are shared by all the videos it does not contain, which is far cheaper. Scores are pooled over all (draw, video) pairs: the expected score with $M$ random examples. Their 95% CI resamples the draws; the range of a **single** draw shows how much one choice of examples can change the result (the split effect).

**Sensitivity to $M$**: the same for several values of $M$, to see how many examples are needed.

In [ ]:
import hashlib
import os
import re
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.realpath("../lib"))
from evaluation import LEVELS, load_labeled_transcripts, plot_confusion
from graph_multimodalsentence import Graph, compute_barycenter
from ot_distances import Fused_Gromov_Wasserstein_distance

## Parameters

In [ ]:
LABELS_FILE = "human_labels.csv"
TEXT_COLUMN = "Transcript"
LABEL_COLUMN = "ability_label_name"
REFERENCE_COLUMN = None         # reference (instructed) level; None: a column whose name contains "ref" or "instruct"
GENERATOR_COLUMN = None         # LLM that generated each video; None: a column whose name contains "model", "generator" or "llm"
DIMENSION = "ability"           # names the outputs

M_VALUES = [1, 2, 3, 5, 7, 10, 15, 20]   # numbers of examples per level tested (too large values are skipped per model)
M_MAIN = 10                     # M of the main results (or the largest M available for a model)
N_DRAWS = 50                    # random draws of the examples, for each model, label and M
SEED = 0
N_BOOTSTRAP = 1000
TOLERANCE = 0.02                # "enough examples": within this of the best score over M

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
ALPHA = 0.14                    # set to the value selected in tune_alpha.ipynb
BARYCENTER_MAX_ITER = 50

OUTPUT_DIR = "results"
CACHE_DIR = os.path.join(OUTPUT_DIR, "per_model_cache")   # distances of each draw, reused at the next run
os.makedirs(CACHE_DIR, exist_ok=True)

## Labeled videos and generator models

In [ ]:
raw = pd.read_csv(LABELS_FILE)
if REFERENCE_COLUMN is None:
    REFERENCE_COLUMN = next((c for c in raw.columns if c not in (TEXT_COLUMN, LABEL_COLUMN)
                             and re.search("ref|instruct", c, re.IGNORECASE)), None)
if GENERATOR_COLUMN is None:
    GENERATOR_COLUMN = next((c for c in raw.columns if re.search("model|generator|llm", c, re.IGNORECASE)), None)
print("reference column:", REFERENCE_COLUMN, "| generator column:", GENERATOR_COLUMN)
assert GENERATOR_COLUMN is not None, "set GENERATOR_COLUMN"

df = load_labeled_transcripts(LABELS_FILE, TEXT_COLUMN, LABEL_COLUMN, REFERENCE_COLUMN)
df["generator"] = df["id"].map(dict(zip([f"row{k}" for k in raw.index], raw[GENERATOR_COLUMN].astype(str))))
LABELS = {"human": "level", **({"reference": "reference"} if REFERENCE_COLUMN else {})}  # label -> column of df
GENERATORS = sorted(df["generator"].unique())

# videos per model, label and level; a level must keep at least one video outside the draw to be evaluated
counts = pd.concat({label: df.groupby("generator")[column].value_counts().unstack().reindex(columns=LEVELS).fillna(0)
                    for label, column in LABELS.items()}, axis=1).astype(int)
max_m = {(g, label): int(counts.loc[g, label].min()) - 1 for g in GENERATORS for label in LABELS}
print("largest usable M per model and label:", max_m)
counts

## Graphs

In [ ]:
class CachedEncoder:
    # Wraps a SentenceTransformer so that each description is encoded only once.
    def __init__(self, model):
        self.model = model
        self.cache = {}

    def encode(self, text):
        if text not in self.cache:
            self.cache[text] = self.model.encode(text)
        return self.cache[text]


embedding = CachedEncoder(SentenceTransformer(EMBEDDING_MODEL))
start = time.time()
graphs = {i: Graph.from_sentence(text, embedding) for i, text in zip(df["id"], df["text"])}
print(f"{len(graphs)} graphs built in {time.time() - start:.0f} s")
fgw = Fused_Gromov_Wasserstein_distance(alpha=ALPHA, features_metric="sqeuclidean", method="temporal")

## Draws

For each model, label and $M$: `N_DRAWS` draws of $M$ videos per level, their 3 barycenters (with a fixed random initialization, so a draw is reproducible), and the distances from every video outside the draw to them. Saved in `results/per_model_cache/` under a name that includes the labels and texts used, so a change of the data computes them again.

In [ ]:
def data_hash(part, column):
    return hashlib.sha1(pd.util.hash_pandas_object(part[["id", "text", column]], index=False).values).hexdigest()[:10]


def run_draws(generator, label, m):
    # Distances of the videos outside each draw to its barycenters: DataFrame (draw, id, distance_<level>).
    column = LABELS[label]
    part = df[df["generator"] == generator]
    path = os.path.join(CACHE_DIR, f"{DIMENSION}_{generator}_{label}_m{m}_draws{N_DRAWS}_seed{SEED}_alpha{ALPHA}"
                                   f"_{data_hash(part, column)}.csv")
    if os.path.exists(path):
        return pd.read_csv(path)
    rng = np.random.default_rng([SEED, m, GENERATORS.index(generator), list(LABELS).index(label)])
    rows = []
    for draw in range(N_DRAWS):
        chosen = {level: rng.choice(part.loc[part[column] == level, "id"].to_numpy(), m, replace=False)
                  for level in LEVELS}
        barycenters = []
        for k, level in enumerate(LEVELS):
            np.random.seed([SEED, m, draw, k])  # the barycenter's random initialization
            barycenters.append(compute_barycenter([graphs[i] for i in chosen[level]], alpha=ALPHA,
                                                  max_iter=BARYCENTER_MAX_ITER))
        used = set(np.concatenate(list(chosen.values())))
        for i in part["id"]:
            if i not in used:
                rows.append({"draw": draw, "id": i,
                             **{f"distance_{level}": fgw.graph_d(graphs[i], b) for level, b in zip(LEVELS, barycenters)}})
    result = pd.DataFrame(rows)
    result.to_csv(path, index=False)
    return result


draws = []
for generator in GENERATORS:
    for label in LABELS:
        for m in M_VALUES:
            if m > max_m[(generator, label)]:
                continue
            start = time.time()
            result = run_draws(generator, label, m)
            draws.append(result.assign(generator=generator, examples=label, M=m))
            print(f"{generator}, {label} examples, M = {m}: {time.time() - start:.0f} s")

draws = pd.concat(draws, ignore_index=True)
distances = draws[[f"distance_{level}" for level in LEVELS]].to_numpy()
draws["pred"] = np.array(LEVELS)[distances.argmin(axis=1)]
draws = draws.merge(df[["id", *LABELS.values()]], on="id")

## Scores with $M$ random examples

For each model, examples label, $M$ and label evaluated against: accuracy and quadratic-weighted $\kappa$ over all (draw, video) pairs, their 95% CI over draws (draws resampled), and the 5–95% range of the accuracy of a single draw. The `all` row pools the three models (draw $d$ of each model together).

In [ ]:
INDEX = {level: k for k, level in enumerate(LEVELS)}
WEIGHTS = np.subtract.outer(np.arange(3), np.arange(3)) ** 2 / 4


def confusion(true, pred):
    return np.bincount(np.asarray(true) * 3 + np.asarray(pred), minlength=9).reshape(3, 3)


def accuracy_of(cm):
    return np.trace(cm) / cm.sum() if cm.sum() else np.nan


def kappa_of(cm):
    expected = np.outer(cm.sum(axis=1), cm.sum(axis=0)) / cm.sum()
    denominator = (WEIGHTS * expected).sum()
    return 1 - (WEIGHTS * cm).sum() / denominator if denominator else np.nan


def summarize_draws(part, target, rng):
    # Pooled scores and their spread over draws, from one confusion matrix per draw.
    cms = np.array([confusion(d[target].map(INDEX), d["pred"].map(INDEX)) for _, d in part.groupby("draw")])
    single = [accuracy_of(cm) for cm in cms]
    boot = [cms[rng.integers(0, len(cms), len(cms))].sum(axis=0) for _ in range(N_BOOTSTRAP)]
    total = cms.sum(axis=0)
    return {"n draws": len(cms), "pairs": int(total.sum()),
            "accuracy": accuracy_of(total),
            "accuracy low": np.percentile([accuracy_of(b) for b in boot], 2.5),
            "accuracy high": np.percentile([accuracy_of(b) for b in boot], 97.5),
            "single draw p5": np.nanpercentile(single, 5), "single draw p95": np.nanpercentile(single, 95),
            "kappa": kappa_of(total),
            "kappa low": np.nanpercentile([kappa_of(b) for b in boot], 2.5),
            "kappa high": np.nanpercentile([kappa_of(b) for b in boot], 97.5)}


rng = np.random.default_rng(SEED)
rows = []
for (label, m), part in draws.groupby(["examples", "M"]):
    full = part.groupby("generator")["draw"].nunique()
    for generator in [*GENERATORS, "all"]:
        if generator == "all":
            if len(full) < len(GENERATORS):  # "all" only when every model has this M
                continue
            subset = part
        else:
            subset = part[part["generator"] == generator]
            if subset.empty:
                continue
        for target, column in LABELS.items():
            rows.append({"generator": generator, "examples": label, "evaluated against": target, "M": m,
                         **summarize_draws(subset, column, rng)})
scores = pd.DataFrame(rows)
scores.to_csv(os.path.join(OUTPUT_DIR, f"human_fgw_per_model_scores_{DIMENSION}.csv"), index=False)
scores.round(3)

## Sensitivity to the number of examples

Accuracy and $\kappa$ as a function of $M$ (examples per level), one line per model; the band is the 95% CI over draws. Rows: label of the examples; columns: label evaluated against.

In [ ]:
def plot_sensitivity(metric, ylabel, chance):
    sources, targets = list(LABELS), list(LABELS)
    fig, axes = plt.subplots(len(sources), len(targets), figsize=(5.5 * len(targets), 3.8 * len(sources)),
                             squeeze=False, sharex=True, sharey=True)
    for r, source in enumerate(sources):
        for c, target in enumerate(targets):
            ax = axes[r, c]
            for generator in [*GENERATORS, "all"]:
                s = scores[(scores["examples"] == source) & (scores["evaluated against"] == target)
                           & (scores["generator"] == generator)].sort_values("M")
                if s.empty:
                    continue
                style = dict(color="black", lw=2) if generator == "all" else {}
                line, = ax.plot(s["M"], s[metric], marker="o", ms=3, label=generator, **style)
                ax.fill_between(s["M"], s[f"{metric} low"], s[f"{metric} high"], color=line.get_color(), alpha=0.15)
            ax.axhline(chance, ls="--", c="grey", lw=1)
            ax.set_title(f"examples: {source}, evaluated against {target}", fontsize=10)
            ax.set_xlabel("M (examples per level)")
            ax.set_ylabel(ylabel)
    axes[0, -1].legend(title="generator", fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"human_fgw_per_model_sensitivity_{metric}_{DIMENSION}.png"), dpi=300)
    plt.show()


plot_sensitivity("accuracy", "accuracy", 1 / len(LEVELS))
plot_sensitivity("kappa", "quadratic-weighted kappa", 0)

### How many examples are enough?

For each model, examples label and label evaluated against: the smallest $M$ whose score is within `TOLERANCE` of the best score over the tested $M$, and the spread of a single draw at that $M$. If the best score is reached at the largest $M$ available, more examples might still help.

In [ ]:
rows = []
for keys, s in scores.groupby(["generator", "examples", "evaluated against"]):
    s = s.sort_values("M")
    best = s["accuracy"].max()
    enough = s[s["accuracy"] >= best - TOLERANCE].iloc[0]
    best_kappa = s["kappa"].max()
    enough_kappa = s[s["kappa"] >= best_kappa - TOLERANCE].iloc[0]
    rows.append({**dict(zip(["generator", "examples", "evaluated against"], keys)),
                 "best accuracy": best, "M enough (accuracy)": int(enough["M"]),
                 "single draw range at that M": f"{enough['single draw p5']:.2f}-{enough['single draw p95']:.2f}",
                 "best kappa": best_kappa, "M enough (kappa)": int(enough_kappa["M"]),
                 "largest M tested": int(s["M"].max()),
                 "still rising": bool(s["accuracy"].idxmax() == s.index[-1] and len(s) > 1)})
enough_table = pd.DataFrame(rows)
enough_table.to_csv(os.path.join(OUTPUT_DIR, f"human_fgw_per_model_enough_{DIMENSION}.csv"), index=False)
enough_table.round(3)

## Main results with `M_MAIN` examples

For each model, the scores with `M_MAIN` examples per level (or the largest $M$ available for that model), and the confusion matrices over all (draw, video) pairs.

In [ ]:
def main_m(generator, label):
    available = sorted(draws.loc[(draws["generator"] == generator) & (draws["examples"] == label), "M"].unique())
    return max([m for m in available if m <= M_MAIN], default=available[0] if available else None)


main_keys = {(g, label): main_m(g, label) for g in GENERATORS for label in LABELS}
main_keys = {key: m for key, m in main_keys.items() if m is not None}
main = pd.concat([draws[(draws["generator"] == g) & (draws["examples"] == label) & (draws["M"] == m)]
                  for (g, label), m in main_keys.items()], ignore_index=True)
main_scores = scores.merge(pd.DataFrame([{"generator": g, "examples": label, "M": m} for (g, label), m in main_keys.items()]),
                           on=["generator", "examples", "M"])
main_scores["accuracy [95% CI]"] = [f"{a:.2f} [{lo:.2f}, {hi:.2f}]" for a, lo, hi in
                                    zip(main_scores["accuracy"], main_scores["accuracy low"], main_scores["accuracy high"])]
main_scores["kappa [95% CI]"] = [f"{k:.2f} [{lo:.2f}, {hi:.2f}]" for k, lo, hi in
                                 zip(main_scores["kappa"], main_scores["kappa low"], main_scores["kappa high"])]
print("M used:", main_keys)
main_scores.pivot_table(index=["examples", "evaluated against"], columns="generator",
                        values=["accuracy [95% CI]", "kappa [95% CI]"], aggfunc="first")

In [ ]:
plot_confusion({(f"vs {target}", f"examples: {source}"): part.assign(true=part[column])
                for target, column in LABELS.items() for source, part in main.groupby("examples")},
               "true", os.path.join(OUTPUT_DIR, f"human_fgw_per_model_confusion_{DIMENSION}.png"))
plt.show()

## Does each LLM respect the requested level?

For each model: the share of its videos whose level found by FGW equals the reference level, with examples labeled by humans or by the reference (`M_MAIN` examples, over all draws; 95% CI over draws), and the share whose **human** level equals the reference level (95% CI over videos). $\chi^2$: equal shares across models, with one prediction per video for FGW (the level whose barycenters are nearest on average over the draws).

In [ ]:
if "reference" not in LABELS:
    print("This section needs a reference level (REFERENCE_COLUMN).")
else:
    rows, tests = [], []
    for source in LABELS:
        s = main_scores[(main_scores["examples"] == source) & (main_scores["evaluated against"] == "reference")]
        for _, r in s.iterrows():
            rows.append({"mode": f"examples: {source}", "generator": r["generator"], "accuracy": r["accuracy"],
                         "low": r["accuracy low"], "high": r["accuracy high"]})
        # chi2 on one prediction per video (the level nearest on average over the draws): the (draw, video)
        # pairs are not independent, a video appearing in many draws
        part = main[main["examples"] == source]
        per_video = part.groupby(["generator", "id", "reference"])[[f"distance_{level}" for level in LEVELS]].mean()
        per_video["pred"] = np.array(LEVELS)[per_video.to_numpy().argmin(axis=1)]
        per_video = per_video.reset_index()
        table = pd.crosstab(per_video["generator"], per_video["pred"] == per_video["reference"])
        if table.shape[1] == 2 and len(table) > 1:
            tests.append({"mode": f"examples: {source}", "p (equal across models)": chi2_contingency(table)[1]})
    rng = np.random.default_rng(SEED)
    for generator in GENERATORS:
        correct = (df.loc[df["generator"] == generator, "level"] == df.loc[df["generator"] == generator, "reference"]).to_numpy(float)
        boot = [correct[rng.integers(0, len(correct), len(correct))].mean() for _ in range(N_BOOTSTRAP)]
        rows.append({"mode": "humans", "generator": generator, "accuracy": correct.mean(),
                     "low": np.percentile(boot, 2.5), "high": np.percentile(boot, 97.5)})
    tests.append({"mode": "humans", "p (equal across models)":
                  chi2_contingency(pd.crosstab(df["generator"], df["level"] == df["reference"]))[1]})
    respect = pd.DataFrame(rows)
    respect.to_csv(os.path.join(OUTPUT_DIR, f"human_fgw_per_model_respect_{DIMENSION}.csv"), index=False)
    display(respect.assign(value=[f"{a:.2f} [{lo:.2f}, {hi:.2f}]" for a, lo, hi in
                                  zip(respect["accuracy"], respect["low"], respect["high"])])
            .pivot(index="mode", columns="generator", values="value"))
    display(pd.DataFrame(tests).round(4))

    mode_order = ["humans", *[f"examples: {source}" for source in LABELS]]
    width = 0.8 / len(GENERATORS)
    fig, ax = plt.subplots(figsize=(9.5, 4))
    for k, generator in enumerate(GENERATORS):
        part = respect[respect["generator"] == generator].set_index("mode").reindex(mode_order)
        x = np.arange(len(mode_order)) + (k - (len(GENERATORS) - 1) / 2) * width
        ax.bar(x, part["accuracy"], width, label=generator,
               yerr=[part["accuracy"] - part["low"], part["high"] - part["accuracy"]], capsize=3)
    ax.axvline(0.5, c="black", lw=0.8)
    ax.axhline(1 / len(LEVELS), ls="--", c="grey", lw=1, label="chance")
    ax.set_xticks(np.arange(len(mode_order)), mode_order)
    ax.set_ylabel("level found = requested level")
    ax.set_title(f"graph + FGW per model: does each LLM respect the requested {DIMENSION} level?")
    ax.legend(title="generator", fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, f"human_fgw_per_model_respect_{DIMENSION}.png"), dpi=300)
    plt.show()